In [ ]:
import matplotlib.animation as animation
import matplotlib.pyplot as plt
import numpy as np

# =====================================================================
# 1. SIMULATION PARAMETERS
# =====================================================================
np.random.seed(42)
N = 12               # number of particles
L = 10.0             # box side length
timesteps = 1000     # number of steps to simulate
dt = 0.01            # time step

# Spring parameters
k = 15.0             # spring stiffness constant
r0 = 0.8             # equilibrium / preferred spacing
rc = 2.0             # neighbor cutoff radius

pos = np.random.uniform(1.0, L - 1.0, size=(N, 2))
vel = np.random.uniform(-4.0, 4.0, size=(N, 2))
mass = np.ones(N)    # all particles have mass 1

positions = np.zeros((timesteps, N, 2))

# =====================================================================
# 2. HELPER FUNCTIONS
# =====================================================================
def pairwise_distances(pos, rc):
    """Broadcasting version (no loops): returns d, dist, neighbors."""
    d = pos[:, None, :] - pos[None, :, :]          # (N, N, 2) vectors r_i - r_j
    dist = np.linalg.norm(d, axis=2)               # (N, N) distances r_ij
    neighbors = (dist > 0) & (dist <= rc)          # True where 0 < r_ij <= rc
    return d, dist, neighbors


def pair_forces(pos):
    """Spring force F_ij = -k (r - r0) * unit vector (pointing from j to i)."""
    d, dist, neighbors = pairwise_distances(pos, rc)
    u = d / (dist[:, :, None] + 1e-12)             # unit vectors
    fmag = -k * (dist - r0) * neighbors            # zero outside the cutoff
    return np.sum(fmag[:, :, None] * u, axis=1)    # sum forces per particle

# =====================================================================
# 3. TRAJECTORY GENERATOR
# =====================================================================
for t in range(timesteps):
    positions[t] = pos.copy()

    # --- Total Force (springs only) ---
    F = pair_forces(pos)

    # --- Time Integration (Euler-Cromer) ---
    vel += (F / mass[:, None]) * dt
    pos += vel * dt

    # --- Wall Reflections ---
    for i in range(N):
        for dim in range(2):
            if pos[i, dim] <= 0 or pos[i, dim] >= L:
                vel[i, dim] *= -1.0
                pos[i, dim] = np.clip(pos[i, dim], 0, L)

# =====================================================================
# 4. SINGLE-PANEL ANIMATION SETUP
# =====================================================================
fig, ax_box = plt.subplots(figsize=(7, 6.5))
fig.suptitle("Atomic Toy Model: Particle Trajectory with Springs",
             fontsize=14, fontweight="bold")

ax_box.set_xlim(-0.5, L + 0.5)
ax_box.set_ylim(-0.5, L + 0.5)
ax_box.set_aspect("equal")

box = plt.Rectangle((0, 0), L, L, fill=False, edgecolor="black", lw=2.5)
ax_box.add_patch(box)

scat = ax_box.scatter(
    positions[0, :, 0],
    positions[0, :, 1],
    color="royalblue",
    s=100,
    edgecolor="black"
)

hud_text = ax_box.text(
    0.02, 0.93, "", transform=ax_box.transAxes,
    fontsize=9, fontfamily="monospace",
    bbox=dict(boxstyle="round", facecolor="white", alpha=0.8)
)

# =====================================================================
# 5. UPDATE FUNCTION FOR FUNCANIMATION
# =====================================================================
def update(frame):
    scat.set_offsets(positions[frame])
    hud_text.set_text(f"Step: {frame:03d}/{timesteps}\nTime: {frame*dt:.2f}s")
    return scat, hud_text

# =====================================================================
# 6. RENDER AND EXPORT ANIMATION FILE
# =====================================================================
plt.tight_layout()
ani = animation.FuncAnimation(
    fig=fig, func=update, frames=timesteps, interval=25, blit=True
)

ani.save("particle_box_simulation.gif", writer="pillow", fps=30)
plt.close()